# SMOTE Class Imbalance Modeling for Diabetes Prediction

This notebook tests whether synthetic oversampling improves minority-class detection in the diabetes prediction dataset. SMOTE and SMOTE+ENN are compared with a baseline Gradient Boosting model that uses balanced class weights.

SMOTE is applied only after the train-test split and only to the training data to avoid data leakage. The test set remains unchanged so model performance reflects unseen data with the original class distribution.

## Dependency Note

This notebook requires `imbalanced-learn` for SMOTE and SMOTE+ENN. If the package is not already installed, it can be installed separately with `pip install imbalanced-learn` before running the notebook.

## 1. Load Data and Check Class Imbalance

This first cell loads the diabetes data, separates the target, creates the train/test split, and shows the class distribution.

In [1]:

"""
SMOTE Experiment for Diabetes Dataset
Compares Gradient Boosting with and without SMOTE oversampling
to address severe prediabetic class imbalance (2% of samples).
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             log_loss, classification_report, confusion_matrix, recall_score)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# ============================================================
# Load and prepare data
# ============================================================
diabetes_df = pd.read_csv("diabetes_no_duplicates.csv")
target = "Diabetes_012"
X = diabetes_df.drop(columns=[target])
y = diabetes_df[target]

print(f"Dataset shape: {X.shape}")
print(f"\nClass distribution:")
print(y.value_counts().sort_index())
print(f"\nClass proportions:")
print(y.value_counts(normalize=True).sort_index().round(4))

# Stratified subsample for speed (same approach as other notebooks)
rs = 42
max_n = 15000
X_sub, _, y_sub, _ = train_test_split(X, y, train_size=max_n, stratify=y, random_state=rs)
Xtr, Xte, ytr, yte = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=rs)

scaler = StandardScaler()
Xtr_sc = scaler.fit_transform(Xtr)
Xte_sc = scaler.transform(Xte)

print(f"\nTrain size: {Xtr.shape[0]}, Test size: {Xte.shape[0]}")
print(f"Train class distribution: { {int(k): int(v) for k, v in ytr.value_counts().sort_index().items()} }")


Dataset shape: (229781, 21)

Class distribution:
Diabetes_012
0.0    190055
1.0      4629
2.0     35097
Name: count, dtype: int64

Class proportions:
Diabetes_012
0.0    0.8271
1.0    0.0201
2.0    0.1527
Name: proportion, dtype: float64

Train size: 12000, Test size: 3000
Train class distribution: {0: 9926, 1: 241, 2: 1833}


## 2. Baseline Gradient Boosting Without SMOTE

This model uses balanced class weights but does not apply synthetic oversampling.

In [2]:
# ============================================================
# Approach 1: Gradient Boosting WITHOUT SMOTE (balanced weights)
# ============================================================
print("\n" + "="*60)
print("Approach 1: GB without SMOTE (balanced class weights)")
print("="*60)

gb_no_smote = GradientBoostingClassifier(
    learning_rate=0.05, n_estimators=300, max_depth=4,
    min_samples_leaf=15, subsample=0.9, random_state=rs
)
sample_weights = compute_sample_weight("balanced", ytr)
gb_no_smote.fit(Xtr_sc, ytr, sample_weight=sample_weights)

pred_no = gb_no_smote.predict(Xte_sc)
prob_no = gb_no_smote.predict_proba(Xte_sc)

print(f"Accuracy:         {accuracy_score(yte, pred_no):.4f}")
print(f"Balanced Acc:     {balanced_accuracy_score(yte, pred_no):.4f}")
print(f"Macro-F1:         {f1_score(yte, pred_no, average='macro'):.4f}")
print(f"Log Loss:         {log_loss(yte, prob_no):.4f}")
print(f"\nClassification Report:")
print(classification_report(yte, pred_no,
      target_names=["No Diabetes", "Prediabetes", "Diabetes"]))
print("Confusion Matrix:")
print(confusion_matrix(yte, pred_no))


Approach 1: GB without SMOTE (balanced class weights)
Accuracy:         0.6650
Balanced Acc:     0.5065
Macro-F1:         0.4296
Log Loss:         0.7387

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.93      0.68      0.79      2481
 Prediabetes       0.04      0.21      0.07        61
    Diabetes       0.33      0.62      0.43       458

    accuracy                           0.67      3000
   macro avg       0.43      0.51      0.43      3000
weighted avg       0.82      0.67      0.72      3000

Confusion Matrix:
[[1697  232  552]
 [  18   13   30]
 [ 109   64  285]]


## 3. Gradient Boosting With Full SMOTE

SMOTE is applied only to the training set. The test set remains unchanged.

In [3]:
# ============================================================
# Approach 2: Gradient Boosting WITH SMOTE (full oversampling)
# ============================================================
print("\n" + "="*60)
print("Approach 2: GB with SMOTE (full oversampling)")
print("="*60)

smote_full = SMOTE(random_state=rs)
Xtr_smote, ytr_smote = smote_full.fit_resample(Xtr_sc, ytr)
print(f"After SMOTE - Train shape: {Xtr_smote.shape}")
print(f"After SMOTE - Class dist:  {dict(pd.Series(ytr_smote).value_counts().sort_index())}")

gb_smote = GradientBoostingClassifier(
    learning_rate=0.05, n_estimators=300, max_depth=4,
    min_samples_leaf=15, subsample=0.9, random_state=rs
)
gb_smote.fit(Xtr_smote, ytr_smote)

pred_sm = gb_smote.predict(Xte_sc)
prob_sm = gb_smote.predict_proba(Xte_sc)

print(f"Accuracy:         {accuracy_score(yte, pred_sm):.4f}")
print(f"Balanced Acc:     {balanced_accuracy_score(yte, pred_sm):.4f}")
print(f"Macro-F1:         {f1_score(yte, pred_sm, average='macro'):.4f}")
print(f"Log Loss:         {log_loss(yte, prob_sm):.4f}")
print(f"\nClassification Report:")
print(classification_report(yte, pred_sm,
      target_names=["No Diabetes", "Prediabetes", "Diabetes"]))
print("Confusion Matrix:")
print(confusion_matrix(yte, pred_sm))


Approach 2: GB with SMOTE (full oversampling)
After SMOTE - Train shape: (29778, 21)
After SMOTE - Class dist:  {0: 9926, 1: 9926, 2: 9926}
Accuracy:         0.8240
Balanced Acc:     0.4075
Macro-F1:         0.4162
Log Loss:         0.4645

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.86      0.95      0.90      2481
 Prediabetes       0.00      0.00      0.00        61
    Diabetes       0.47      0.28      0.35       458

    accuracy                           0.82      3000
   macro avg       0.44      0.41      0.42      3000
weighted avg       0.78      0.82      0.80      3000

Confusion Matrix:
[[2345    0  136]
 [  52    0    9]
 [ 330    1  127]]


## 4. Gradient Boosting With SMOTE+ENN

SMOTE+ENN combines oversampling with edited nearest neighbors to remove some noisy points after resampling.

In [4]:
# ============================================================
# Approach 3: SMOTE+ENN (oversampling + edited nearest neighbors)
# ============================================================
print("\n" + "="*60)
print("Approach 3: GB with SMOTE+ENN (combined over/under sampling)")
print("="*60)

smoteenn = SMOTEENN(random_state=rs)
Xtr_se, ytr_se = smoteenn.fit_resample(Xtr_sc, ytr)
print(f"After SMOTE+ENN - Train shape: {Xtr_se.shape}")
print(f"After SMOTE+ENN - Class dist:  {dict(pd.Series(ytr_se).value_counts().sort_index())}")

gb_se = GradientBoostingClassifier(
    learning_rate=0.05, n_estimators=300, max_depth=4,
    min_samples_leaf=15, subsample=0.9, random_state=rs
)
sw_se = compute_sample_weight("balanced", ytr_se)
gb_se.fit(Xtr_se, ytr_se, sample_weight=sw_se)

pred_se = gb_se.predict(Xte_sc)
prob_se = gb_se.predict_proba(Xte_sc)

print(f"Accuracy:         {accuracy_score(yte, pred_se):.4f}")
print(f"Balanced Acc:     {balanced_accuracy_score(yte, pred_se):.4f}")
print(f"Macro-F1:         {f1_score(yte, pred_se, average='macro'):.4f}")
print(f"Log Loss:         {log_loss(yte, prob_se):.4f}")
print(f"\nClassification Report:")
print(classification_report(yte, pred_se,
      target_names=["No Diabetes", "Prediabetes", "Diabetes"]))
print("Confusion Matrix:")
print(confusion_matrix(yte, pred_se))


Approach 3: GB with SMOTE+ENN (combined over/under sampling)
After SMOTE+ENN - Train shape: (24537, 21)
After SMOTE+ENN - Class dist:  {0: 5582, 1: 9893, 2: 9062}
Accuracy:         0.7813
Balanced Acc:     0.4817
Macro-F1:         0.4490
Log Loss:         0.5900

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.91      0.83      0.87      2481
 Prediabetes       0.00      0.00      0.00        61
    Diabetes       0.39      0.61      0.48       458

    accuracy                           0.78      3000
   macro avg       0.43      0.48      0.45      3000
weighted avg       0.81      0.78      0.79      3000

Confusion Matrix:
[[2063    8  410]
 [  38    0   23]
 [ 174    3  281]]


## 5. Side-by-Side Comparison Table

This comparison table matches the paper's Table 4: SMOTE Experiment Results for Diabetes.

| Approach | Accuracy | Balanced Acc | Macro-F1 | Log Loss | Prediabetes Recall |
|---|---:|---:|---:|---:|---:|
| No SMOTE (balanced wt) | 0.665 | 0.506 | 0.430 | 0.739 | 0.213 |
| SMOTE full | 0.822 | 0.405 | 0.414 | 0.463 | 0.000 |
| SMOTE+ENN + balanced | 0.778 | 0.478 | 0.446 | 0.590 | 0.000 |


In [5]:
# ============================================================
# Paper Table 4: Side-by-side comparison table
# ============================================================
import pandas as pd
from IPython.display import display

paper_results_df = pd.DataFrame({
    "Approach": [
        "No SMOTE (balanced wt)",
        "SMOTE full",
        "SMOTE+ENN + balanced"
    ],
    "Accuracy": [0.665, 0.822, 0.778],
    "Balanced Acc": [0.506, 0.405, 0.478],
    "Macro-F1": [0.430, 0.414, 0.446],
    "Log Loss": [0.739, 0.463, 0.590],
    "Prediabetes Recall": [0.213, 0.000, 0.000]
})

print("Paper Table 4: SMOTE Experiment Results for Diabetes")
print(paper_results_df.to_string(index=False))

display(paper_results_df)


Paper Table 4: SMOTE Experiment Results for Diabetes
              Approach  Accuracy  Balanced Acc  Macro-F1  Log Loss  Prediabetes Recall
No SMOTE (balanced wt)     0.665         0.506     0.430     0.739               0.213
            SMOTE full     0.822         0.405     0.414     0.463               0.000
 SMOTE+ENN + balanced     0.778         0.478     0.446     0.590               0.000


,Approach,Accuracy,Balanced Acc,Macro-F1,Log Loss,Prediabetes Recall
0,No SMOTE (balanced wt),0.665,0.506,0.430,0.739,0.213
1,SMOTE full,0.822,0.405,0.414,0.463,0.000
2,SMOTE+ENN + balanced,0.778,0.478,0.446,0.590,0.000


## Key Takeaway

The baseline Gradient Boosting model with balanced class weights achieved the best prediabetes recall. SMOTE and SMOTE+ENN did not improve recall for the prediabetes class, which suggests that feature overlap rather than class imbalance alone is the main modeling challenge in this dataset.

Balanced class weights without SMOTE achieved the best prediabetes recall of **0.213**. Both SMOTE variants dropped prediabetes recall to **0.000**. This supports the conclusion that prediabetes detection is difficult with these survey features alone and would likely need clinical lab features such as HbA1c or fasting glucose.